In [6]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd

df = pd.read_csv("elo.csv")
TEAM_ELO = dict(zip(df["team"], df["elo"]))
print(TEAM_ELO)

sorted_teams = sorted(
    TEAM_ELO.items(),
    key=lambda x: x[1],
    reverse=True
)

POTS = {
    1: sorted_teams[0:12],
    2: sorted_teams[12:24],
    3: sorted_teams[24:36],
    4: sorted_teams[36:48]
}

CONFEDERATIONS = {
    "UEFA": ["Germany", "England", "Spain", "Portugal", "Netherlands", "Sweden", "Belgium", 
             "France", "Croatia", "Türkiye", "Czechia", "Switzerland", "Norway", "Austria", "Scotland", "Bosnia & Herzegovina"],
    "CONMEBOL": ["Brazil", "Argentina", "Colombia", "Paraguay", "Ecuador", "Uruguay"],
    "CONCACAF": ["Mexico", "Canada", "USA", "Panama", "Curaçao", "Haiti"],
    "CAF": ["South Africa", "Morocco", "Tunisia", "Egypt", "Côte d'Ivoire", "Senegal", "Algeria", "Cabo Verde", "Congo DR", "Ghana"],
    "AFC": ["Korea Republic", "Qatar", "Australia", "Japan", "IR Iran", "Saudi Arabia", "Uzbekistan", "Iraq", "Jordan"],
    "OFC": ["New Zealand"]
}

def solve_group_creation(TEAM_ELO, POTS, CONFEDERATIONS):
    model = gp.Model("group_creation")

    groups = range(12)
    teams = list(TEAM_ELO.keys())

    x = model.addVars(groups, teams, vtype=GRB.BINARY, name="x")

    for team in teams:
        model.addConstr(gp.quicksum(x[g, team] for g in groups) == 1)

    for g in groups:
        for pot in POTS:
            pot_teams = [team for team, elo in POTS[pot]]
            model.addConstr(gp.quicksum(x[g, team] for team in pot_teams) == 1)

    for g in groups:
        for conf, conf_teams in CONFEDERATIONS.items():
            max_teams = 2 if conf == "UEFA" else 1
            model.addConstr(gp.quicksum(x[g, team] for team in conf_teams) <= max_teams)

    max_strength = model.addVar(vtype=GRB.CONTINUOUS, name="max_strength")
    min_strength = model.addVar(vtype=GRB.CONTINUOUS, name="min_strength")

    for g in groups:
        group_strength = gp.quicksum(TEAM_ELO[team] * x[g, team] for team in teams)
        model.addConstr(group_strength <= max_strength)
        model.addConstr(group_strength >= min_strength)

    model.setObjective(max_strength - min_strength, GRB.MINIMIZE)
    model.optimize()

    if model.status == GRB.OPTIMAL:
        print("\n===== OPTIMIZED WORLD CUP GROUPS =====")

        schedule_rows = []

        for g in groups:
            group_teams = [team for team in teams if x[g, team].X > 0.5]
            group_teams = sorted(group_teams, key=lambda t: TEAM_ELO[t], reverse=True)
            group_strength = sum(TEAM_ELO[team] for team in group_teams)

            group = chr(65 + g)

            print(f"\nGroup {group}:")
            for team in group_teams:
                print(f"  {team} (ELO: {TEAM_ELO[team]:.2f})")
            print(f"  Total ELO: {group_strength:.2f}")

            schedule_rows.append({
                "group": group,
                "teams": "; ".join(group_teams)
            })

        schedule_df = pd.DataFrame(schedule_rows)
        schedule_df.to_csv("optimized_groups.csv", index=False)

        print("\n======================================")
        print(f"Fairness objective: {model.ObjVal:.2f}")
        print("Saved to optimized_groups.csv")

solve_group_creation(TEAM_ELO, POTS, CONFEDERATIONS)

{'Mexico': 1687.48, 'Canada': 1559.48, 'USA': 1671.23, 'Germany': 1735.77, 'England': 1828.02, 'Brazil': 1765.86, 'Spain': 1874.71, 'Portugal': 1767.85, 'Argentina': 1877.27, 'Colombia': 1698.35, 'South Africa': 1428.38, 'Korea Republic': 1591.63, 'Czechia': 1505.74, 'Bosnia & Herzegovina': 1387.22, 'Qatar': 1450.31, 'Switzerland': 1650.06, 'Morocco': 1755.1, 'Haiti': 1293.1, 'Scotland': 1503.34, 'Paraguay': 1505.35, 'Australia': 1579.34, 'Türkiye': 1605.73, 'Curaçao': 1294.77, "Côte d'Ivoire": 1540.87, 'Ecuador': 1598.52, 'Netherlands': 1753.57, 'Japan': 1661.58, 'Sweden': 1509.79, 'Tunisia': 1476.41, 'Belgium': 1742.24, 'Egypt': 1562.37, 'IR Iran': 1619.58, 'New Zealand': 1275.58, 'Cabo Verde': 1371.11, 'Saudi Arabia': 1423.88, 'Uruguay': 1673.07, 'France': 1870.7, 'Senegal': 1684.07, 'Iraq': 1446.28, 'Norway': 1557.44, 'Algeria': 1571.03, 'Austria': 1597.4, 'Jordan': 1387.74, 'Congo DR': 1105.96, 'Uzbekistan': 1458.73, 'Croatia': 1714.87, 'Ghana': 1346.88, 'Panama': 1539.16}
Gurobi 